In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load silver table
df_silver = spark.table("dataanalytics.ml1.loan_book_silver")

# Create 4 new features based on EDA bivariate analysis
df_transformed = df_silver \
    .withColumn("loan_to_income_ratio", 
                F.col("loan_amount") / F.col("annual_income")) \
    .withColumn("debt_servicing_stress_index", 
                F.col("interest_rate") * F.col("dti_ratio")) \
    .withColumn("delinquency_concentration", 
                F.col("num_delinquencies_2yr") / F.col("months_since_oldest_account")) \
    .withColumn("high_earner_low_stability_flag",
                F.when((F.col("annual_income") > 100000) & (F.col("employment_length_years") < 3), 1)
                 .otherwise(0))

# Add 70/30 train/test split
df_with_split = df_transformed.withColumn(
    "data_split",
    F.when(F.rand(seed=42) <= 0.7, "train").otherwise("test")
)

# Select only required features in specified order
final_columns = [
    "annual_income",
    "interest_rate",
    "employment_length_years",
    "num_delinquencies_2yr",
    "months_since_oldest_account",
    "num_open_accounts",
    "loan_amount",
    "dti_ratio",
    "months_since_last_delinquency",
    "pct_accounts_current",
    "high_earner_low_stability_flag",
    "loan_to_income_ratio",
    "debt_servicing_stress_index",
    "delinquency_concentration",
    "default_flag",
    "data_split"
]

df_gold = df_with_split.select(final_columns)

# Save to Gold layer
df_gold.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dataanalytics.ml1.loan_book_gold_v2")